In [1]:
%load_ext autoreload
%autoreload 2

# == Imports ==
import os
import json
import sys
import shutil
import importlib

import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scipy.signal as signal
from datasets import load_dataset
from IPython.display import Audio, display
import jiwer
import platform
import time
import warnings
from pprint import pprint
import pickle
import joblib
from pathlib import Path
import gradio as gr
from itertools import islice
import modal

# Project imports
import asr_pipeline_benchmark.configuration as configuration
importlib.reload(configuration) # Forces the reloading of load_dotenv() for rapid testing. To be turned off in production
from asr_pipeline_benchmark.configuration import Config

from asr_pipeline_benchmark.storage import LocalStorage, GCSStorage

from asr_pipeline_benchmark.models import load_transcriber_engine, run_checkpoint_inference

from asr_pipeline_benchmark.preprocessing import (
    cleanup_pipeline, add_white_noise, add_synthetic_reverberation,
    extract_ir_from_mit_dataset, extract_irs_from_mit_dataset_by_location,
    add_real_reverberation, add_smart_audio_dropouts,
    packet_loss, band_limit, hard_clip_ratio, hard_clip_ratio_norm,
    codec_degrade
)
from asr_pipeline_benchmark.utility import (
    generate_sine_tone, generate_run_label, checkpoint_iterator,
    cleanup_results_col_names, expand_mfcc_results
)
from asr_pipeline_benchmark.audio_analysis import analyze_asr_audio_health
from asr_pipeline_benchmark.data import get_mit_ir_dataset, get_speech_dataset
from asr_pipeline_benchmark.visualization import plot_feature_scatter_grid
from asr_pipeline_benchmark.evaluation import run_framework_speed_comparison

# Filter out the benign duplicate logits processor warnings from transformers
warnings.filterwarnings("ignore", message=".*logits processor of type.*")


# Set figsize global params
plt.rcParams['figure.figsize'] = (12,5)

modalApp = modal.App("asr-benchmark-pipeline")

config = Config()

In [ ]:
# This is what is need to reload .env
# importlib.reload(configuration)
# config = Config()

print(config.USE_CLOUD_STORAGE)
print(config.TRANSCRIBE_CACHE_DIR)
print(config.CHECKPOINT_DIR)

In [2]:
# Quick check on results
# === Verification on sample used in both loops ===
if config.USE_CLOUD_STORAGE:
    storageInterface = GCSStorage(config.GCS_BUCKET_NAME)
else:
    storageInterface = LocalStorage()

pd.DataFrame(storageInterface.load_pickle(str(config.TRANSCRIBE_CACHE_DIR)+"/smallDr__mlx__5230c951__5samples.pkl"))


,idx,model,corr_type,wer,w_hits,w_subs,w_dels,w_ins,num_words,cer,c_hits,c_subs,c_dels,c_ins,num_chars,sample_analysis,ref,hyp
0,6930-75918-0000,mlx-community_whisper-small-mlx,none,0.000000,8,0,0,0,8,0.000000,46,0,0,0,46,"{'metrics': {'rms_lin_global': 0.0294, 'rms_db...",concord returned to its place amidst the tents,concord returned to its place amidst the tents
1,6930-75918-0000,mlx-community_whisper-small-mlx,WN_3snr,0.125000,7,1,0,0,8,0.021739,46,0,0,1,46,"{'metrics': {'rms_lin_global': 0.036, 'rms_dbf...",concord returned to its place amidst the tents,concorde returned to its place amidst the tents
2,6930-75918-0000,mlx-community_whisper-small-mlx,SV_0.5,0.000000,8,0,0,0,8,0.000000,46,0,0,0,46,"{'metrics': {'rms_lin_global': 0.0207, 'rms_db...",concord returned to its place amidst the tents,concord returned to its place amidst the tents
3,6930-75918-0000,mlx-community_whisper-large-v3-turbo,none,0.000000,8,0,0,0,8,0.000000,46,0,0,0,46,"{'metrics': {'rms_lin_global': 0.0294, 'rms_db...",concord returned to its place amidst the tents,concord returned to its place amidst the tents
4,6930-75918-0000,mlx-community_whisper-large-v3-turbo,WN_3snr,0.000000,8,0,0,0,8,0.000000,46,0,0,0,46,"{'metrics': {'rms_lin_global': 0.036, 'rms_dbf...",concord returned to its place amidst the tents,concord returned to its place amidst the tents
5,6930-75918-0000,mlx-community_whisper-large-v3-turbo,SV_0.5,0.000000,8,0,0,0,8,0.000000,46,0,0,0,46,"{'metrics': {'rms_lin_global': 0.0207, 'rms_db...",concord returned to its place amidst the tents,concord returned to its place amidst the tents
6,6930-75918-0001,mlx-community_whisper-small-mlx,none,0.023256,42,1,0,0,43,0.026087,224,2,4,0,230,"{'metrics': {'rms_lin_global': 0.0385, 'rms_db...",the english forwarded to the french baskets of...,the english voted to the french baskets of flo...
7,6930-75918-0001,mlx-community_whisper-small-mlx,WN_3snr,0.023256,42,1,0,0,43,0.026087,224,2,4,0,230,"{'metrics': {'rms_lin_global': 0.0471, 'rms_db...",the english forwarded to the french baskets of...,the english voted to the french baskets of flo...
8,6930-75918-0001,mlx-community_whisper-small-mlx,SV_0.5,0.023256,42,1,0,0,43,0.026087,224,2,4,0,230,"{'metrics': {'rms_lin_global': 0.0288, 'rms_db...",the english forwarded to the french baskets of...,the english voted to the french baskets of flo...
9,6930-75918-0001,mlx-community_whisper-large-v3-turbo,none,0.000000,43,0,0,0,43,0.000000,230,0,0,0,230,"{'metrics': {'rms_lin_global': 0.0385, 'rms_db...",the english forwarded to the french baskets of...,the english forwarded to the french baskets of...


# Load dataset

In [ ]:
# == Load dataset in stream mode ==

print(f"Loading LibriSpeech dataset stream...")
# dataset = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True)
dataset = get_speech_dataset()

In [ ]:
# == real room IR reverberation ==
# ir_ds = load_dataset("benjamin-paine/mit-impulse-response-survey", split="train")
ir_ds = get_mit_ir_dataset()

location_id2label = ir_ds.features["location"].int2str
location_str2id = ir_ds.features["location"].str2int
ir_ds_locations = ir_ds.features['location'].names
#pprint(ir_ds_locations)

## Quick tests on data

In [ ]:
# == Grab first sample ==
iterator = iter(dataset)
sample = next(iterator)
# sample2 = next(iterator)
# sample2 = next(iterator)
audio_array = sample["audio"]["array"]
sample_rate = sample["audio"]["sampling_rate"]
ground_truth_text = sample['text']

print(f"Sample Rate: {sample_rate} Hz")
print(f"Ground truth text: {ground_truth_text}")

In [ ]:
sample = next(iterator)
sample

In [ ]:
# == Plot the clean waveform ==
librosa.display.waveshow(audio_array, sr=sample_rate)
plt.title('Clean Speech Waveform - Sample 1')
plt.show()

In [ ]:
# == Listen to the audio ==
Audio(audio_array, rate=sample_rate)

# Audio Modification/Corruption Functions

### White Noise tests

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_0dB = add_white_noise(audio_array, 0)
    pprint(analyze_asr_audio_health(test_signal_0dB))
    librosa.display.waveshow(test_signal_0dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({0} dB SNR White Noise)")
    plt.legend()
    plt.show()

    Audio(test_signal_0dB, rate=sample_rate)

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_3dB = add_white_noise(audio_array, 3)
    pprint(analyze_asr_audio_health(test_signal_3dB))
    librosa.display.waveshow(test_signal_3dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({3} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_3dB, rate=sample_rate)


In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_6dB = add_white_noise(audio_array, 6)
    pprint(analyze_asr_audio_health(test_signal_6dB))
    librosa.display.waveshow(test_signal_6dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({6} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_6dB, rate=sample_rate)


In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_9dB = add_white_noise(audio_array, 9)
    pprint(analyze_asr_audio_health(test_signal_9dB))
    librosa.display.waveshow(test_signal_9dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({9} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_9dB, rate=sample_rate)

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_12dB = add_white_noise(audio_array, 12)
    pprint(analyze_asr_audio_health(test_signal_12dB))
    librosa.display.waveshow(test_signal_12dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({12} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_12dB, rate=sample_rate)

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_20dB = add_white_noise(audio_array, 20)
    pprint(analyze_asr_audio_health(test_signal_20dB))
    librosa.display.waveshow(test_signal_20dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({20} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_20dB, rate=sample_rate)

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_neg3dB = add_white_noise(audio_array, -3)
    pprint(analyze_asr_audio_health(test_signal_neg3dB))
    librosa.display.waveshow(test_signal_neg3dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({-3} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_neg3dB, rate=sample_rate)


In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_neg6dB = add_white_noise(audio_array, -6)
    pprint(analyze_asr_audio_health(test_signal_neg6dB))
    librosa.display.waveshow(test_signal_neg6dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({-6} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_neg6dB, rate=sample_rate)

### Inspection of different normlization effect on IR RMS and dynamic

In [ ]:
irs = extract_irs_from_mit_dataset_by_location(ir_ds, location_str2id("Stairwell"))
print("".join([f"id: {e['id']} \ndetail: {e['detail']} \nduration in secs: {e['duration']} \n---------\n\n" for e in irs]))

In [ ]:
test_ir = irs[1]['audio']
librosa.display.waveshow(audio_array, color='blue', alpha=0.5)
librosa.display.waveshow(test_ir, color='red', alpha=0.5)

In [ ]:
l_inf = test_ir / np.max(np.abs(test_ir))
librosa.display.waveshow(audio_array, color='blue', alpha=0.5)
librosa.display.waveshow(l_inf, color='red', alpha=0.5)

In [ ]:
l2 = test_ir / np.sqrt(np.sum(test_ir**2))
librosa.display.waveshow(audio_array, color='blue', alpha=0.5)
librosa.display.waveshow(l_inf, color='red', alpha=0.5)

In [ ]:
rms = np.sqrt(np.mean(test_ir**2))
target_rms = 0.03
ir_scaled = test_ir * (target_rms / rms)
ir_scaled = np.clip(ir_scaled, -1.0, 1.0)
librosa.display.waveshow(audio_array, color='blue', alpha=0.5)
librosa.display.waveshow(ir_scaled, color='red', alpha=0.5)

# CORRUPTION EVALUATION

In [ ]:
# ==  GRADIO LAUNCHER - Packet Loss ==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:

        gr.Markdown(f"Packet loss")
        with gr.Column():
            for val in [(40, .10), (30, .20), (20, .30), (10, .50), (5, .70)]:
                audio = packet_loss(audio_array, sr=16_000, packet_ms=val[0], loss_rate=val[1])
                # Convert float32 [-1.0, 1.0] to int16 [-32786, 32767]
                audio_int16 = (audio * 32767).astype(np.int16)
                gr_audio = (16_000, audio_int16)
                output = gr.Audio(gr_audio, label=f"{val[1]*100}% packet loss of length of {val[0]}ms")

    demo.launch(prevent_thread_lock=True)

In [ ]:
# ==  GRADIO LAUNCHER - Smart Audio Dropouts ==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:

        gr.Markdown(f"Packet loss")
        with gr.Column():
            for num in [5, 10, 15, 20]:
                for dur in [50, 75, 100]:
                    audio = add_smart_audio_dropouts(audio_array, sr=16_000, num_dropouts=num,
                                                     dropout_duration_ms=dur, threshold_below_global_db=15)

                    audio_int16 = (audio * 32767).astype(np.int16)
                    gr_audio = (16_000, audio_int16)
                    output = gr.Audio(gr_audio, label=f"{num} dropouts of {dur}ms each, threshold below {15}dB")

    demo.launch(prevent_thread_lock=True)

In [ ]:
# ==  GRADIO LAUNCHER -  BAND LIMITING ==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:

        gr.Markdown(f"Packet loss")
        with gr.Column():
            for lc, hc in [(800, 3000), (600, 2000), (500, 1000)]:
                audio = band_limit(audio_array, sr=16_000, lowcut=lc, highcut=hc)
                # Convert float32 [-1.0, 1.0] to int16 [-32786, 32767]
                audio_int16 = (audio * 32767).astype(np.int16)
                gr_audio = (16_000, audio_int16)
                output = gr.Audio(gr_audio, label=f"Band limiting on low cut at {lc} and high cut at {hc}")

    demo.launch(prevent_thread_lock=True)

In [ ]:
irs = extract_irs_from_mit_dataset_by_location(ir_ds, location_str2id("Stairwell"))
print("".join([f"id: {e['id']} \ndetail: {e['detail']} \nduration in secs: {e['duration']} \n---------\n\n" for e in irs]))

In [ ]:
# ==  GRADIO LAUNCHER - REAL REVERB ==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:
        n = len(irs)
        gr.Markdown(f"# {n} different IRs applied to audio")
        with gr.Column():
            for ir in irs:
                audio = add_real_reverberation(audio_array, ir["audio"])
                audio_int16 = (audio * 32767).astype(np.int16)
                gr_audio = (16_000, audio_int16)
                output = gr.Audio(gr_audio, label=f"ID:{ir["id"]} | {location_id2label(ir['location'])}: {ir['detail']} | {round(ir['duration'],2)} secs")

    demo.launch(prevent_thread_lock=True)

In [ ]:
# ==  GRADIO LAUNCHER - HARD AND SOFT CLIPPING==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:
        gr.Markdown(f"# Hard clip and normzalized hard clip applied to audio on different thresholds")
        with gr.Column():
            for th in [.5, .3, .2, .1, .05, .025]:
                audio1 = hard_clip_ratio(audio_array, threshold=th)
                audio1_int16 = (audio1 * 32767).astype(np.int16)
                gr_audio1 = (16_000, audio1_int16)
                output = gr.Audio(gr_audio1, label=f"Hard clip at {th} threshold")

                audio2 = hard_clip_ratio_norm(audio_array, threshold=th)
                audio2_int16 = (audio2 * 32767).astype(np.int16)
                gr_audio2 = (16_000, audio2_int16)
                output = gr.Audio(gr_audio2, label=f"Hard clip normalized at {th} threshold")

    demo.launch(prevent_thread_lock=True)

In [ ]:
# === GRADIO LAUCHER - Bitrate / Codec degradation

gr.close_all()

# Each codec has its own legal bitrate range — Vorbis can't go below ~45k.
BITRATE_MATRIX = {
    "libopus":    ["6k", "8k"]
}

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:
        gr.Markdown(f"# Bitrate / Codec degradation")
        with gr.Column():
            for codec, bitrates in BITRATE_MATRIX.items():
                for bitrate in bitrates:
                    try:
                        audio = codec_degrade(audio_array, sr=16_000, bitrate=bitrate, codec=codec)
                    except (ValueError, RuntimeError) as e:
                        # Don't let one unencodable pair tear down the whole demo.
                        print(f"Skipping {codec} @ {bitrate}: {e}")
                        continue

                    audio_int16 = (audio * 32767).astype(np.int16)
                    gr_audio = (16_000, audio_int16)
                    output = gr.Audio(gr_audio, label=f"{codec} at {bitrate} bitrate")

    demo.launch(prevent_thread_lock=True)

The target RMS normalization with clipping worked the best. This is the normalization method implemented in the add_real_reverberation function

In [ ]:
gr.close_all()

# Run Benchmark Tests

In [ ]:
# USE THIS AS A CORRUPTION LIST METHOD TO KEEP TRACK OF PARAMETERS USED
ir, _ = extract_ir_from_mit_dataset(ir_idx=251)

corruptions = {
    "none": lambda x: x,
    "WN_3snr": lambda x: add_white_noise(x, 3),
    "SV_0.5": lambda x: add_synthetic_reverberation(x, mix=0.5),
    "RV_0.5_251": lambda x: add_real_reverberation(x, ir, mix=0.5),
    "SDO_8_100_15": lambda x: add_smart_audio_dropouts(x, num_dropouts=8, dropout_duration_ms=100, threshold_below_global_db=15),
    "WN_3-SV_.5-SDO_8_100_15": lambda x: add_smart_audio_dropouts(
                                                add_synthetic_reverberation(
                                                    add_white_noise(x, 3),
                                                mix=0.5),
                                          num_dropouts=8, dropout_duration_ms=100, threshold_below_global_db=15),
    "codec_8k": lambda x: codec_degrade(x, bitrate="8k"),
    "codec_6k": lambda x: codec_degrade(x, bitrate="6k"),
    "PL_40_.1": lambda x: packet_loss(x, packet_ms=40 , loss_rate=0.10),
    "PL_30_.2": lambda x: packet_loss(x, packet_ms=30 , loss_rate=0.20),
    "PL_20_.3": lambda x: packet_loss(x, packet_ms=20 , loss_rate=0.30),
    "PL_10_.4": lambda x: packet_loss(x, packet_ms=10 , loss_rate=0.40),
    "PL_5_.5": lambda x: packet_loss(x, packet_ms=5 , loss_rate=0.50),
    "BL_800_3000": lambda x: band_limit(x, lowcut=800, highcut=3000),
    "BL_600_2000": lambda x: band_limit(x, lowcut=600, highcut=2000),
    "BL_500_1000": lambda x: band_limit(x, lowcut=500, highcut=1000),
    "clip_1": lambda x: hard_clip_ratio(x, 0.1),
    "clip_05": lambda x: hard_clip_ratio(x, 0.05),
    "clip_025": lambda x: hard_clip_ratio(x, 0.025),
    "clipN_1": lambda x: hard_clip_ratio_norm(x, 0.1),
    "clipN_05": lambda x: hard_clip_ratio_norm(x, 0.05),
    "clipN_025": lambda x: hard_clip_ratio_norm(x, 0.025),
}

In [ ]:
# # === TEMPORARY SMALL CORRUPTIONS LIST FOR TESTING ===
# corruptions = {
#     "none": lambda x: x,
#     "WN_3snr": lambda x: add_white_noise(x, 3),
#     "SV_0.5": lambda x: add_synthetic_reverberation(x, mix=0.5),
# }

In [ ]:
# == RUN DRESS REHEARSAL BENCHMARK TEST ON 10 SAMPELS ==
if config.USE_CLOUD_STORAGE:
    storageInterface = GCSStorage(config.GCS_BUCKET_NAME)
else:
    storageInterface = LocalStorage()

# print(f"Starting inference with storageInterface: {storageInterface.__class__.__name__}")

for run_small_model in [True, False]:
    transcriber = load_transcriber_engine(config.IS_APPLE_SILICON, model_small=run_small_model)
    testtype = "dr"
    n = 10
    run_label = generate_run_label(testtype, get_model_backend(transcriber), n, MODEL_SMALL=run_small_model)
    # corruption_type_list = ["none", "whitenoise", "synth-verb", "real-verb", "dropout", ["whitenoise", "synth-verb", "dropout"]]
    # corruption_type_list = ["none", "whitenoise", "synth-verb", "dropout"]
    iterator = iter(dataset)
    dress_test = run_checkpoint_inference(transcriber, iterator, run_label, n, storageInterface, corruption_type=corruptions, clear_cache=True)


In [ ]:
# === Verification on sample used in both loops ===
if config.USE_CLOUD_STORAGE:
    storageInterface = GCSStorage(config.GCS_BUCKET_NAME)
else:
    storageInterface = LocalStorage()


# dr_run_smallModel = None
# with open(config.TRANSCRIBE_CACHE_DIR / "dr__mlx-community_whisper-small-mlx__4samples.pkl", "rb") as f:
#     dr_run_smallModel = pickle.load(f)
dr_run_smallModel = storageInterface.load_pickle(str(config.TRANSCRIBE_CACHE_DIR)+"/smallDr__mlx-community_whisper-small-mlx__3samples.pkl")

# dr_run_largeModel = None
# with open(config.TRANSCRIBE_CACHE_DIR / "dr__mlx-community_whisper-large-v3-turbo__4samples.pkl", "rb") as f:
#     dr_run_largeModel = pickle.load(f)
dr_run_largeModel = storageInterface.load_pickle(str(config.TRANSCRIBE_CACHE_DIR)+"/smallDr__mlx-community_whisper-large-v3-turbo__3samples.pkl")

# verify that the samples used are the same for the small and large model transcription benchmark run
dr_small = pd.DataFrame(dr_run_smallModel)
dr_large = pd.DataFrame(dr_run_largeModel)

assert dr_small["idx"].equals(dr_large["idx"]), True
assert dr_small["ref"].equals(dr_large["ref"]), True

# pd.concat([dr_small["idx"], dr_large["idx"], dr_small["ref"], dr_large["ref"]], axis=1)

In [ ]:
# == LARGE INGERENCE BENCHMARK TEST ==
if config.USE_CLOUD_STORAGE:
    storageInterface = GCSStorage()
else:
    storageInterface = LocalStorage()

for run_small_model in [True, False]:
    transcriber = load_transcriber_engine(config.IS_APPLE_SILICON, model_small=run_small_model)
    testtype = "benchtest"
    n=1000
    run_label = generate_run_label(testtype, get_model_backend(transcriber), n, MODEL_SMALL=run_small_model)
    iterator = iter(dataset)
    bench_run = run_checkpoint_inference(transcriber, iterator, run_label, n, corruption_type=corruptions, clear_cache=True)


In [ ]:
# === Verification on sample used in both loops ===
bench_run_smallModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "dr__mlx-community_whisper-small-mlx__8samples.pkl", "rb") as f:
    bench_run_smallModel = pickle.load(f)

bench_run_largeModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "dr__mlx-community_whisper-large-v3-turbo__8samples.pkl", "rb") as f:
    bench_run_largeModel = pickle.load(f)

# verify that the samples used are the same for the small and large model transcription benchmark run
df_small = pd.DataFrame(bench_run_smallModel)
df_large = pd.DataFrame(bench_run_largeModel)

assert df_small["idx"].equals(df_large["idx"]), True
assert df_small["ref"].equals(df_large["ref"]), True

# pd.concat([df_small["idx"], df_large["idx"], df_small["ref"], df_large["ref"]], axis=1)

## Load Large Whisper Model Inference Transcription Results

In [ ]:
bench_run_largeModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "benchtest__mlx-community_whisper-large-v3-turbo__1000samples.pkl", "rb") as f:
    bench_run_largeModel = pickle.load(f)

## Load Small Whisper Model Inference Transcription Results

In [ ]:
bench_run_smallModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "benchtest__mlx-community_whisper-small-mlx__1000samples.pkl", "rb") as f:
    bench_run_smallModel = pickle.load(f)

## EDA and Data Engineering

In [ ]:
# verify that the samples used ar ethe same for the small and large model transcription benchmark run
df_small = pd.DataFrame(bench_run_smallModel)
df_large = pd.DataFrame(bench_run_largeModel)

df_small["ref"].equals(df_large["ref"])

pd.concat([df_small["idx"], df_large["idx"], df_small["ref"], df_large["ref"]], axis=1)


In [ ]:
df_small_sorted = df_small.sort_values(by=['idx', 'corr_type'], ascending=True)
df_large_sorted = df_large.sort_values(by=['idx', 'corr_type'], ascending=True)

In [ ]:
pd.concat([df_small_sorted["idx"], df_large_sorted["idx"], df_small_sorted["ref"], df_large_sorted["ref"]], axis=1)

In [ ]:
df_smallModel = cleanup_results_col_names(bench_run_smallModel)
df_largeModel = cleanup_results_col_names(bench_run_largeModel)
# df_smallModel.info()
# df_largeModel.info()

In [ ]:
# == Save DF to JSON file
# df_tests.to_json("./results/bench_test_history.json", orient="records", lines=True)

# Read it back into Pandas DF
# loaded_df = pd.read_json("./results/bench_test_history.json", orient="records", lines=True)
# loaded_df.columns

### Flatten MFCCs

In [ ]:
# === CLeanup DF ===

# Expland MFCCs columns
df_smallModel_exp = expand_mfcc_results(df_smallModel)
df_largeModel_exp = expand_mfcc_results(df_largeModel)

# Convert idx to integers and change col name
df_smallModel_exp["idx"] = pd.factorize(df_smallModel_exp["idx"])[0]
df_smallModel_exp.rename(columns={"idx": "sample_id"})
df_largeModel_exp["idx"] = pd.factorize(df_largeModel_exp["idx"])[0]
df_largeModel_exp.rename(columns={"idx": "sample_id"})

In [ ]:
df_smallModel_exp.info()

In [ ]:
# == generate column lists per type
num_cols = df_smallModel_exp.select_dtypes(include=['number']).columns.tolist()

cat_cols = df_smallModel_exp.select_dtypes(include=['object']).columns.tolist()

print(f"numerical columns: \n\t{num_cols}")
print(f"categorical columns: \n\t{cat_cols}")

In [ ]:
# == Look at cross-correlation between numerical variables ==
cols_to_correlate = [
    'wer',
    'cer',
    'mtr.crest_factor_db',
    'mtr.min_zcr',
    'mtr.std_zcr',
    'mtr.mean_zcr',
    'mtr.rms_lin_global',
    'mtr.rms_dbfs_global',
    'mtr.mfcc1_mean', 'mtr.mfcc2_mean', 'mtr.mfcc3_mean', 'mtr.mfcc4_mean', 'mtr.mfcc5_mean', 'mtr.mfcc6_mean', 'mtr.mfcc7_mean',
    'mtr.mfcc8_mean', 'mtr.mfcc9_mean', 'mtr.mfcc10_mean', 'mtr.mfcc11_mean', 'mtr.mfcc12_mean', 'mtr.mfcc13_mean',
    'mtr.mfcc1_std', 'mtr.mfcc2_std', 'mtr.mfcc3_std', 'mtr.mfcc4_std', 'mtr.mfcc5_std', 'mtr.mfcc6_std', 'mtr.mfcc7_std',
    'mtr.mfcc8_std', 'mtr.mfcc9_std', 'mtr.mfcc10_std', 'mtr.mfcc11_std', 'mtr.mfcc12_std', 'mtr.mfcc13_std',
    'mtr.sf_mean', 'mtr.sf_std'
]

# df_smallModel_exp[cols_to_correlate].info()

corr_df = df_smallModel_exp[cols_to_correlate].dropna()
corr_matrix = corr_df.corr(method='spearman')

plt.figure(figsize=(22, 18))
mask = np.tril(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix,
            mask=mask,
            annot=True,
            fmt='.2f',
            annot_kws={"size":8},
            cmap='coolwarm',
            vmin=-1, vmax=1,
            square=True,
            linewidths=.5,
            cbar_kws={"shrink": .5}
)

# gca -> get current axis
plt.gca().xaxis.tick_top()
plt.xticks(rotation=45, ha="left", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
# plt.tick_params(top=True, left=True, bottom=False, right=False)
plt.title("Correlation: Audio Metrics vs WER Degradation - SMALL MODEL")
plt.tight_layout()
plt.show()


Inpsecting correlation with WER

- CER must be removed, would cause leakage (0.97 correlation)
- RMS lin and DBFS global are 100% correlated. we will keep DBFS
- all other features have a low correlation to WER
- MFCCs standard deviation present a lot of collineraity within themselves. Will clean out anything above 0.6


In [ ]:
# Clean out the correlated features
corr_matrix = corr_matrix.abs()

# Isolate upper triangle of the matrix to avoid evaluating the same pair twice
# Using the upper triangle and dropping col if any values in the upper triangle col is above threshold will only remove features (col) from the right
# In this case here, since MFCC are ordered in terms of columns, this automatically keep the lower order ones.
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(np.bool))

collinearity_threshold = 0.60
redundant_features = [col for col in upper_triangle.columns if any(upper_triangle[col] > collinearity_threshold)]

# Restrict the dropping logic to MFCC standard deviation
mfcc_std_to_drop = [col for col in redundant_features if 'mfcc' in col and 'std' in col]
# pprint(redundant_features)
# pprint(mfcc_std_to_drop)

to_drop_set = set(mfcc_std_to_drop)
to_drop_set.add('cer')
to_drop_set.add('mtr.rms_lin_global')
# to_drop_set

pruned_features = [item for item in cols_to_correlate if item not in to_drop_set]

pprint(pruned_features)

In [ ]:
# ==  scatter plot of delta_wer and delta_zcr ==
fig, axes = plt.subplots(2, 2, figsize=(14, 6))

# Plot 1: Noisy ZCR vs WER
sns.scatterplot(x='mtr.mean_zcr', y='wer', data=df_smallModel, ax=axes[0,0], alpha=0.6)
axes[0,0].set_title('Noisy Mean ZCR vs WER Degradation - Small Model')
axes[0,0].set_xlabel("Noisy Audio ZCR")
axes[0,0].set_ylabel("WER")

# Plot 2: Original RMS vs WER
sns.scatterplot(x='mtr.rms_dbfs_global', y='wer', data=df_smallModel, ax=axes[0,1], alpha=0.6)
axes[0,1].set_title('Original Clean RMS vs WER Degradation - Small Model')
axes[0,1].set_xlabel('Clean RMS (dBFS)')
axes[0,1].set_ylabel('WER')

# Plot 1: Noisy ZCR vs WER
sns.scatterplot(x='mtr.mean_zcr', y='wer', data=df_largeModel, ax=axes[1,0], alpha=0.6)
axes[1,0].set_title('Noisy Mean ZCR vs WER Degradation - Large Model')
axes[1,0].set_xlabel("Noisy Audio ZCR")
axes[1,0].set_ylabel("WER")

# Plot 2: Original RMS vs WER
sns.scatterplot(x='mtr.rms_dbfs_global', y='wer', data=df_largeModel, ax=axes[1,1], alpha=0.6)
axes[1,1].set_title('Original Clean RMS vs WER Degradation - Large Model')
axes[1,1].set_xlabel('Clean RMS (dBFS)')
axes[1,1].set_ylabel('WER')

plt.tight_layout()
plt.show()

In [ ]:
plot_feature_scatter_grid(
    df_smallModel,
    pruned_features,  # 'wer' is filtered out automatically
    target="wer",
    ncols=4,
    title="WER vs pruned features - Small Model",
    # save_path="./images/wer_scatter_small.png",
)


In [ ]:
plot_feature_scatter_grid(
    df_largeModel,
    pruned_features,
    target="wer",
    ncols=4,
    title="WER vs pruned features - Large Model",
)

In [ ]:
# Plot Noist ZCR vs Delta WER with colored dots for RMS dBFS
sns.scatterplot(
    x='mtr.mean_zcr',
    y='wer',
    hue='mtr.min_zcr',
    palette='viridis',
    data=df_smallModel,
    alpha=0.8,
    s=60
)

plt.title('Noisy ZCR vs WER Degradation (Colored by mtr.min_zcr)')
plt.xlabel('Noisy Audio ZCR')
plt.ylabel('WER (Degradation)')

# Move the legend outside the plot
plt.legend(title='min ZCR', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Framework Speed Comparison (MLX vs MPS)

In [ ]:
# == RUN FRAMEWORK COMPARISON TEST ==
if config.RUN_FRAMEWORK_SPEED_COMPARISON_TEST:
    # IS_APPLE_SILICON = platform.system() == "Darwin" and platform.processor() == "arm"
    # IS_APPLE_SILICON= False
    transcriber = load_transcriber_engine()
    run_framework_speed_comparison(dataset, n_samples=10)